# Kaggle: train the detector

Attach as Datasets: the competition data, and the `cell-tracking-src` zip from `scripts/package_for_kaggle.py` (code only, no checkpoint needed here). Set the GPU accelerator on.

In [ ]:
import time
SESSION_STARTED = time.time()

## Dependency preflight and stale-code check

Fails fast, before the slow cache build, if the attached code Dataset is stale or a dependency is missing (a stale Dataset once cost a full cache build before the mismatch surfaced).

In [ ]:
import os
import subprocess, sys
from pathlib import Path

SRC_DATASET = Path('/kaggle/input/cell-tracking-src')  # adjust to the attached Dataset's mount name
assert SRC_DATASET.exists(), f'code Dataset not attached at {SRC_DATASET}'
sys.path.insert(0, str(SRC_DATASET / 'src'))
# subprocess.run below spawns a FRESH Python process that does not
# inherit this sys.path.insert -- it needs PYTHONPATH in its own env.
SRC_ENV = {**os.environ, 'PYTHONPATH': str(SRC_DATASET / 'src')}

# Unlike kaggle_run.ipynb (the actual submission notebook, which runs with
# internet disabled and does not need this), this training notebook is never
# submitted -- it just produces a checkpoint you download. A plain internet
# install is fine here; make sure Settings > Internet is ON for this session.
for pkg in ('blosc2', 'zarr', 'zstandard'):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], check=True)

import torch
print('cuda:', torch.cuda.is_available())

from cell_tracking import config
print('on_kaggle:', config.on_kaggle())
print('train_dir:', config.get_train_dir())


In [ ]:
# 1. Build the prepared-frame cache (once per session).
subprocess.run([sys.executable, str(SRC_DATASET / 'scripts' / 'build_cache.py'),
                '--cache-dir', '/kaggle/working/cache'], check=True, env=SRC_ENV)

In [ ]:
# 2. Train. max_hours below the session wall; started_at accounts for the cache build above.
#
# epochs=90 is a real target, not a nominal cap: T_max for the cosine LR
# schedule is epochs, independent of max_hours/patience (those only decide
# WHEN the run stops, not how the LR is paced). At the ~6.4 min/epoch this
# model measured on Kaggle, 90 epochs is close to what the ~10h training
# budget can actually reach, so the LR anneals across the run it will
# actually get instead of sitting flat near its initial value throughout
# (confirmed last run: lr was still 0.0009996 of 0.001 at epoch 6).
#
# val_frames_per_volume=16 (up from the 4-frame default): at 4 frames x 20
# held-out volumes = 80 frames total, val_loss swung by ~2x epoch to epoch
# on pure sampling noise, which is what made patience=5 trigger on noise
# rather than a real generalization signal last run.
from cell_tracking.train import History, train

history = History()
ckpt = train(
    config.get_train_dir(),
    Path('/kaggle/working/detector.pt'),
    epochs=90,
    val_frames_per_volume=16,
    cache_dir=Path('/kaggle/working/cache'),
    history=history,
    max_hours=11.0,
    started_at=SESSION_STARTED,
    patience=15,
    resume=False,  # explicit clean start -- do not pick up a leftover /kaggle/working/detector.pt
)
print('checkpoint:', ckpt)
